In [1]:
import pandas as pd
import numpy as np

# Load data
raw = pd.read_csv('/Users/wiktor/TIC/Spring2026-TIC/data/merged_prices_weather.csv')

raw['date'] = pd.to_datetime(raw['date'])

# Identify column groups (price is equal across locations, so we can separate it from location-specific weather)
price_cols   = ['corn_Close', 'corn_Volume', 'corn_log_return',
                'soybean_Close', 'soybean_Volume', 'soybean_log_return']
meta_cols    = ['date', 'Latitude', 'Longitude', 'Location_ID']
weather_cols = [c for c in raw.columns if c not in price_cols + meta_cols]

# Price table: one row per date (prices are identical across locations)
prices = (
    raw.groupby('date')[price_cols]
       .first()
       .reset_index()
)

# Weather table: pivot to wide format(one row per date, separate columns for each location)
raw_weather = raw.dropna(subset=['Location_ID']).copy()
raw_weather['Location_ID'] = raw_weather['Location_ID'].astype(int)
weather_wide = raw.pivot(index='date', columns='Location_ID', values=weather_cols)
weather_wide.columns = [f"{var}_loc{loc}" for var, loc in weather_wide.columns]
weather_wide = weather_wide.reset_index()

# Merge
data = prices.merge(weather_wide, on='date')

# Set date index & deduplicate
data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data[~data.index.duplicated(keep='first')]
data = data.sort_index()

print(f"Wide dataset shape: {data.shape}")
print(f"Date range: {data.index.min().date()} → {data.index.max().date()}")
print(f"Columns ({len(data.columns)} total):")
print(data.columns.tolist()[:10], "...")  # preview first 10
data.head(3)

Wide dataset shape: (5305, 162)
Date range: 2005-01-03 → 2026-02-02
Columns (162 total):
['corn_Close', 'corn_Volume', 'corn_log_return', 'soybean_Close', 'soybean_Volume', 'soybean_log_return', 'temperature_2m_mean_locnan', 'temperature_2m_mean_loc1.0', 'temperature_2m_mean_loc2.0', 'temperature_2m_mean_loc3.0'] ...


,corn_Close,corn_Volume,corn_log_return,soybean_Close,soybean_Volume,soybean_log_return,temperature_2m_mean_locnan,temperature_2m_mean_loc1.0,temperature_2m_mean_loc2.0,temperature_2m_mean_loc3.0,...,soil_temperature_0_to_7cm_mean_loc2.0,soil_temperature_0_to_7cm_mean_loc3.0,soil_temperature_28_to_100cm_mean_locnan,soil_temperature_28_to_100cm_mean_loc1.0,soil_temperature_28_to_100cm_mean_loc2.0,soil_temperature_28_to_100cm_mean_loc3.0,soil_temperature_7_to_28cm_mean_locnan,soil_temperature_7_to_28cm_mean_loc1.0,soil_temperature_7_to_28cm_mean_loc2.0,soil_temperature_7_to_28cm_mean_loc3.0
date,,,,,,,,,,,,,,,,,,,,,
2005-01-03,201.75,1743.0,0.000000,537.25,47,0.000000,NaN,11.580083,9.055416,6.513833,...,10.184585,5.168000,NaN,5.542584,13.367916,6.813833,NaN,8.771749,10.949168,5.303416
2005-01-04,201.00,1357.0,-0.003724,529.75,54,-0.014058,NaN,4.525917,7.024166,8.643000,...,8.736667,7.170083,NaN,6.175917,13.167916,6.861749,NaN,7.267583,10.107500,6.636750
2005-01-05,201.50,1159.0,0.002484,534.00,37,0.007991,NaN,0.390500,7.551250,5.203417,...,8.428333,4.782584,NaN,6.390499,12.880416,6.942999,NaN,5.090500,9.442917,5.793000


In [ ]:
import statsmodels.api as sm
import statsmodels.tsa.stattools as ts
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Rolling Window Beta Estimation

WINDOW = 400  # 1 trading year

print("Estimating beta with 1-year rolling window (no look-ahead bias)...")
print(f"Window size: {WINDOW} trading days\n")

# Prepare log prices
log_corn = np.log(data['corn_Close'])
log_soy = np.log(data['soybean_Close'])

# Drop any NaNs or zeros
valid_idx = (data['cvorn_Close'] > 0) & (data['soybean_Close'] > 0)
log_corn = log_corn[valid_idx]
log_soy = log_soy[valid_idx]

print(f"Total rows available: {len(log_corn)}")

# Initialize results
data['beta'] = np.nan
data['alpha'] = np.nan
data['ecm_spread'] = np.nan

# Rolling window estimation
for i in range(WINDOW, len(log_corn)):
    # Training window: PAST 252 days (not including current day; no look-ahead)
    train_log_corn = log_corn.iloc[i-WINDOW:i]
    train_log_soy = log_soy.iloc[i-WINDOW:i]
    
    # Estimate beta and alpha on training window
    X = sm.add_constant(train_log_soy)
    try:
        ols = sm.OLS(train_log_corn, X).fit()
        beta_t = ols.params.iloc[1]  # FIX: use iloc[1] for slope
        alpha_t = ols.params.iloc[0]  # FIX: use iloc[0] for intercept
    except:
        # If OLS fails, use previous values
        beta_t = data['beta'].iloc[i-1] if i > WINDOW else np.nan
        alpha_t = data['alpha'].iloc[i-1] if i > WINDOW else np.nan
    
    # Store parameters for this date
    current_idx = log_corn.index[i]
    data.loc[current_idx, 'beta'] = beta_t
    data.loc[current_idx, 'alpha'] = alpha_t
    
    # Compute spread using beta/alpha from PAST data only
    data.loc[current_idx, 'ecm_spread'] = (
        log_corn.iloc[i] - beta_t * log_soy.iloc[i] - alpha_t
    )

print(f"✓ Beta estimation complete. First {WINDOW} rows have NaN (insufficient history).\n")

# Summary Statistics

valid_beta = data['beta'].dropna()

print("=" * 80)
print("BETA STATISTICS (Rolling 1-Year Window)")
print("=" * 80)
print(f"Mean:   {valid_beta.mean():.4f}")
print(f"Median: {valid_beta.median():.4f}")
print(f"Std:    {valid_beta.std():.4f}")
print(f"Min:    {valid_beta.min():.4f}")
print(f"Max:    {valid_beta.max():.4f}")
print(f"Range:  {valid_beta.max() - valid_beta.min():.4f}")

# Cointegration Test on the Spread

valid_spread = data['ecm_spread'].dropna()

adf_stat, adf_p, _, _, _, _ = ts.adfuller(valid_spread)
print(f"\n" + "=" * 80)
print("COINTEGRATION TEST (on rolling-window spread)")
print("=" * 80)
print(f"ADF Statistic: {adf_stat:.4f}")
print(f"P-Value:       {adf_p:.5f}")
if adf_p < 0.05:
    print("Result: COINTEGRATION DETECTED — spread is mean-reverting ✓")
else:
    print("Result: NO COINTEGRATION at 5% level ✗")

# Visualization

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: Beta over time
ax1.plot(data.index, data['beta'], linewidth=1.5, label='Rolling beta (252 days)', color='steelblue')
ax1.fill_between(data.index, 
                  valid_beta.mean() - valid_beta.std(),
                  valid_beta.mean() + valid_beta.std(),
                  alpha=0.2, color='steelblue', label='±1 std')
ax1.set_ylabel('Hedge Ratio (β)', fontsize=11)
ax1.set_title('Beta Evolution — 1-Year Rolling Window (No Look-Ahead Bias)', fontsize=12, fontweight='bold')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# Bottom: ECM Spread
ax2.plot(data.index, data['ecm_spread'], linewidth=1, color='darkgreen', alpha=0.8)
ax2.axhline(0, color='black', linestyle='--', linewidth=1)
ax2.fill_between(data.index, -valid_spread.std(), valid_spread.std(), 
                  alpha=0.2, color='gray', label='±1 std')
ax2.set_xlabel('Date', fontsize=11)
ax2.set_ylabel('ECM Spread', fontsize=11)
ax2.set_title('ECM Residual Spread (Corn − β·Soy − α)', fontsize=12)
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()